# 01 — EDA e Qualidade

Este notebook faz a inspeção inicial do dataset: shape, tipos, nulos, cardinalidade e distribuições de preço, desconto e rating. Também exporta um relatório de qualidade para `reports/`.

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_DIR = Path.cwd().resolve()
while PROJECT_DIR.name != 'amazon-product-intelligence' and PROJECT_DIR.parent != PROJECT_DIR:
    PROJECT_DIR = PROJECT_DIR.parent

DATA_RAW = PROJECT_DIR / 'data' / 'raw' / 'dados_amazon.csv'
REPORTS_DIR = PROJECT_DIR / 'reports'
FIGURES_DIR = REPORTS_DIR / 'figures'
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(PROJECT_DIR / 'notebooks'))
from src.preprocessing import load_raw_dataset

sns.set_theme(style='whitegrid')


## Carregamento e inspeção geral

In [ ]:
df_raw = load_raw_dataset(DATA_RAW)
df_raw.shape


In [ ]:
df_raw.head(3)


In [ ]:
dtypes = df_raw.dtypes.astype(str).to_frame('dtype')
dtypes


## Completude e missingness

In [ ]:
missing_pct = (df_raw.isna().mean() * 100).sort_values(ascending=False)
missing_pct.to_frame('missing_%').head(20)


In [ ]:
plt.figure(figsize=(12, 6))
sns.heatmap(df_raw.isna(), cbar=False)
plt.title('Missingness Map (row × column)')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'missingness_map.png', dpi=160)
plt.close()


## Distribuições principais

In [ ]:
plt.figure(figsize=(10, 4))
sns.histplot(df_raw['rating'].dropna(), bins=30, kde=True)
plt.title('Distribution: rating')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'dist_rating.png', dpi=160)
plt.close()


In [ ]:
def _clean_currency(s: pd.Series) -> pd.Series:
    return (
        s.astype(str)
        .str.replace('₹', '', regex=False)
        .str.replace(',', '', regex=False)
        .replace({'nan': np.nan, 'None': np.nan, '': np.nan})
        .astype(float)
    )

discounted = _clean_currency(df_raw['discounted_price'])
actual = _clean_currency(df_raw['actual_price'])
discount_pct = (
    df_raw['discount_percentage']
    .astype(str)
    .str.replace('%', '', regex=False)
    .str.replace(',', '', regex=False)
    .replace({'nan': np.nan, 'None': np.nan, '': np.nan})
    .astype(float)
)

plt.figure(figsize=(10, 4))
sns.histplot(discounted.dropna(), bins=40, kde=False)
plt.title('Distribution: discounted_price (cleaned)')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'dist_discounted_price.png', dpi=160)
plt.close()

plt.figure(figsize=(10, 4))
sns.histplot(discount_pct.dropna(), bins=40, kde=True)
plt.title('Distribution: discount_percentage (cleaned)')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'dist_discount_pct.png', dpi=160)
plt.close()


## Distribuição por categoria

In [ ]:
main_category = df_raw['category'].astype(str).str.split('|').str[0].replace({'nan': np.nan})
cat_counts = main_category.value_counts().head(15)
cat_counts


In [ ]:
plt.figure(figsize=(12, 5))
sns.barplot(x=cat_counts.index, y=cat_counts.values)
plt.xticks(rotation=45, ha='right')
plt.title('Top Categories (main_category)')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'products_by_category.png', dpi=160)
plt.close()


## Exportar relatório de qualidade

In [ ]:
critical_cols = ['product_id', 'product_name', 'category', 'discounted_price', 'actual_price', 'discount_percentage', 'rating', 'rating_count']
critical_missing = (df_raw[critical_cols].isna().mean() * 100).sort_values(ascending=False)

report_lines = []
report_lines.append('# Data Quality Report')
report_lines.append('')
report_lines.append(f'- Rows: {len(df_raw):,}')
report_lines.append(f'- Columns: {df_raw.shape[1]}')
report_lines.append('')
report_lines.append('## Missingness (Top 20)')
for col, pct in missing_pct.head(20).items():
    report_lines.append(f'- {col}: {pct:.2f}%')
report_lines.append('')
report_lines.append('## Critical Fields Missingness')
for col, pct in critical_missing.items():
    report_lines.append(f'- {col}: {pct:.2f}%')

(REPORTS_DIR / 'data_quality_report.md').write_text("\\n".join(report_lines), encoding='utf-8')
REPORTS_DIR / 'data_quality_report.md'
